In [1]:
# ============================================================================
# BLOCK 7: ENGAGEMENT METRICS - ROMANIAN
# Input: 06_umap_data_ro.pkl
# Output: 07_engagement_data_ro.pkl + CSV exports
# Runtime: ~3 minutes
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 11:58:18
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
print('='*70)
print('BLOCK 7: ENGAGEMENT METRICS - ROMANIAN')
print('='*70)

if check_checkpoint_exists('ro', '07_engagement_data'):
    df_ro = load_checkpoint('ro', '07_engagement_data')
    print('✓ Loading from engagement checkpoint')
else:
    df_ro = load_checkpoint('ro', '06_umap_data')
    if df_ro is None:
        raise FileNotFoundError('Run 05_umap_visualization_ro.ipynb first')

# Engagement by sentiment
print('\n--- ENGAGEMENT BY SENTIMENT ---\n')

engagement_sentiment = df_ro.groupby('sentiment').agg({
    'comment_likes': ['mean', 'median', 'sum', 'count'],
    'replies': ['mean', 'median', 'sum']
}).round(2)

print('Likes by Sentiment:')
display(engagement_sentiment['comment_likes'])
print('\nReplies by Sentiment:')
display(engagement_sentiment['replies'])

# Engagement by topic
print('\n--- ENGAGEMENT BY TOPIC ---\n')

engagement_topic = df_ro.groupby('topic').agg({
    'comment_likes': ['mean', 'median', 'sum'],
    'replies': ['mean', 'median', 'sum'],
    'clean_text': 'count'
}).round(2)
engagement_topic.columns = ['likes_mean', 'likes_median', 'likes_sum', 
                            'replies_mean', 'replies_median', 'replies_sum',
                            'comment_count']
engagement_topic = engagement_topic.sort_values('comment_count', ascending=False)

print('Top Topics by Engagement:')
display(engagement_topic.head(8))

# Cross-tab
print('\n--- SENTIMENT × TOPIC CROSS-TAB ---\n')

cross_tab = pd.crosstab(df_ro['topic'], df_ro['sentiment'], 
                        values=df_ro['comment_likes'], aggfunc='mean').round(2)
display(cross_tab)

# Heatmap
print('\nGenerating engagement heatmap...')

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(cross_tab, annot=True, fmt='.0f', cmap='YlOrRd', 
            cbar_kws={'label': 'Average Likes'})
ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Topic', fontsize=12)
ax.set_title('Average Likes by Topic × Sentiment - Romanian', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'ro_engagement_heatmap.png', 'ro', 'visualizations')

# Export
engagement_df = df_ro.groupby('sentiment').agg({
    'comment_likes': 'mean',
    'replies': 'mean',
    'clean_text': 'count'
}).reset_index()
engagement_df.columns = ['sentiment', 'avg_likes', 'avg_replies', 'comment_count']
engagement_df.to_csv(OUTPUT_DIR / 'romanian' / 'ro_engagement_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'romanian' / 'ro_engagement_summary.csv'}")

final_cols = ['video_id', 'author_name', 'comment_text', 'clean_text', 
              'comment_date', 'comment_likes', 'replies',
              'sentiment', 'topic', 'topic_probability', 'umap_x', 'umap_y']
final_cols = [c for c in final_cols if c in df_ro.columns]

df_ro[final_cols].to_csv(OUTPUT_DIR / 'romanian' / 'ro_final_dataset.csv', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'romanian' / 'ro_final_dataset.csv'}")

save_checkpoint(df_ro, 'ro', '07_engagement_data')
update_pipeline_status('ro', 7, 'completed')

print('\n' + '='*70)
print('✓ BLOCK 7 COMPLETE - ROMANIAN PIPELINE FINISHED')
print('='*70)
print(f'\nAll Romanian outputs saved to: {OUTPUT_DIR / "romanian"}')
print('='*70)


BLOCK 7: ENGAGEMENT METRICS - ROMANIAN
✓ Loading checkpoint: 06_umap_data_ro.pkl

--- ENGAGEMENT BY SENTIMENT ---

Likes by Sentiment:


,mean,median,sum,count
sentiment,,,,
Negative,17.56,0.0,83834.0,4774
Positive,46.67,1.0,85217.0,1826



Replies by Sentiment:


,mean,median,sum
sentiment,,,
Negative,1.25,0.0,5947.0
Positive,2.71,0.0,4946.0



--- ENGAGEMENT BY TOPIC ---

Top Topics by Engagement:


,likes_mean,likes_median,likes_sum,replies_mean,replies_median,replies_sum,comment_count
topic,,,,,,,
-1.0,26.53,1.0,95326.0,1.78,0.0,6389.0,3593
0.0,35.41,1.0,41963.0,1.86,0.0,2199.0,1185
1.0,17.73,0.0,13705.0,0.96,0.0,741.0,773
2.0,14.76,1.0,8649.0,1.40,0.0,823.0,586
3.0,19.53,1.0,6837.0,1.57,0.0,548.0,350
4.0,41.09,0.0,1849.0,2.42,0.0,109.0,45
5.0,17.15,2.0,686.0,1.88,0.0,75.0,40
6.0,1.29,1.0,36.0,0.32,0.0,9.0,28



--- SENTIMENT × TOPIC CROSS-TAB ---



sentiment,Negative,Positive
topic,,
-1.0,16.56,52.97
0.0,26.74,55.85
1.0,15.22,25.33
2.0,10.07,25.77
3.0,11.96,39.02
4.0,54.27,4.83
5.0,17.75,14.75
6.0,1.42,0.50



Generating engagement heatmap...
✓ Saved: outputs\ro\visualizations\ro_engagement_heatmap.png

✓ Saved: outputs\romanian\ro_engagement_summary.csv
✓ Saved: outputs\romanian\ro_final_dataset.csv
✓ Checkpoint saved: 07_engagement_data_ro.pkl
✓ Pipeline status updated: ro - Block 7 - completed

✓ BLOCK 7 COMPLETE - ROMANIAN PIPELINE FINISHED

All Romanian outputs saved to: outputs\romanian
